# Retail Sales Forecasting and Customer Analytics
## 02 - Data Cleaning and Preparation

This notebook prepares the Olist e-commerce datasets for further analysis. The cleaning decisions are based on the issues identified during the initial data inspection. The original raw datasets are preserved and all cleaning and transformation steps are performed on separate DataFrames.

In [2]:
## Import pandas and locate the raw data
import pandas as pd
from pathlib import Path

raw_data_path = Path("../01_Raw_Data")
cleaned_data_path = Path("../02_Cleaned_Data")

print("Raw data folder:", raw_data_path)
print("Cleaned data folder:", cleaned_data_path)

Raw data folder: ..\01_Raw_Data
Cleaned data folder: ..\02_Cleaned_Data


In [3]:
customers = pd.read_csv(raw_data_path / "olist_customers_dataset.csv")
geolocation = pd.read_csv(raw_data_path / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(raw_data_path / "olist_order_items_dataset.csv")
payments = pd.read_csv(raw_data_path / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(raw_data_path / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(raw_data_path / "olist_orders_dataset.csv")
products = pd.read_csv(raw_data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_data_path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    raw_data_path / "product_category_name_translation.csv"
)

print("All raw datasets loaded successfully.")

All raw datasets loaded successfully.


## Date and Timestamp Conversion

Date and timestamp columns were imported from the CSV files as strings. These fields are converted to datetime format to support time-based analysis, calculation of delivery periods and sales forecasting.

In [4]:
# Orders date columns
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

# Order items date column
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

# Reviews date columns
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(reviews[column], errors="coerce")

print("Date and timestamp columns converted successfully.")

Date and timestamp columns converted successfully.


In [5]:
## Verify the conversion
print("Orders:")
print(orders[order_date_columns].dtypes)

print("\nOrder Items:")
print(order_items[["shipping_limit_date"]].dtypes)

print("\nReviews:")
print(reviews[review_date_columns].dtypes)

Orders:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Order Items:
shipping_limit_date    datetime64[us]
dtype: object

Reviews:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [6]:
## Validate the date conversion
print("Missing values after date conversion")
print("-" * 50)

print("\nOrders:")
print(orders[order_date_columns].isnull().sum())

print("\nOrder Items:")
print(order_items[["shipping_limit_date"]].isnull().sum())

print("\nReviews:")
print(reviews[review_date_columns].isnull().sum())

Missing values after date conversion
--------------------------------------------------

Orders:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order Items:
shipping_limit_date    0
dtype: int64

Reviews:
review_creation_date       0
review_answer_timestamp    0
dtype: int64


## Preparation of Completed Orders

The raw orders dataset contains several order statuses, including delivered, shipped, cancelled, unavailable, invoiced and processing orders. For completed sales analysis, a separate dataset containing delivered orders is created. The original orders DataFrame is retained so that non-completed orders remain available for other analyses.

In [7]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

print("Total orders:", len(orders))
print("Delivered orders:", len(delivered_orders))
print("Excluded from completed sales dataset:", len(orders) - len(delivered_orders))

Total orders: 99441
Delivered orders: 96478
Excluded from completed sales dataset: 2963


In [8]:
## Investigate missing dates within delivered orders
print("Missing dates within delivered orders")
print("-" * 50)

print(
    delivered_orders[
        [
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ].isnull().sum()
)

Missing dates within delivered orders
--------------------------------------------------
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [9]:
## Check missing product categories in actual delivered sales
delivered_order_items = order_items[
    order_items["order_id"].isin(delivered_orders["order_id"])
].copy()

delivered_items_products = delivered_order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

print("Order items from delivered orders:", len(delivered_order_items))

print(
    "Delivered order items with missing product category:",
    delivered_items_products["product_category_name"].isnull().sum()
)

print(
    "Unique products with missing category in delivered orders:",
    delivered_items_products.loc[
        delivered_items_products["product_category_name"].isnull(),
        "product_id"
    ].nunique()
)

Order items from delivered orders: 110197
Delivered order items with missing product category: 1537
Unique products with missing category in delivered orders: 584


## Product Category Treatment

Some products with missing category information are present in completed customer orders. Removing these records would also remove valid transaction values from the analysis. Therefore, missing product categories are retained and labelled as `Unknown` for category-based analysis.

In [10]:
products["product_category_name"] = (
    products["product_category_name"].fillna("Unknown")
)

print("Missing product categories after treatment:")
print(products["product_category_name"].isnull().sum())

print("\nProducts labelled as Unknown:")
print((products["product_category_name"] == "Unknown").sum())

Missing product categories after treatment:
0

Products labelled as Unknown:
610


In [11]:
## Add the English product category
products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

print("Products after category translation merge:")
print(products.shape)

print("\nMissing English category names:")
print(products["product_category_name_english"].isnull().sum())

Products after category translation merge:
(32951, 10)

Missing English category names:
623


In [12]:
## Find the untranslated categories
untranslated = products[
    products["product_category_name_english"].isnull()
]

print("Original categories without English translation:")
print(
    untranslated["product_category_name"]
    .value_counts()
)

print("\nNumber of affected products:")
print(len(untranslated))

Original categories without English translation:
product_category_name
Unknown                                          610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64

Number of affected products:
623


## Product Category Standardisation

The supplied category translation table does not contain an English translation for every product category. To avoid removing valid products or manually introducing unsupported translations, a new analysis category field is created. The English category name is used where available, while the original category name is retained where no translation is provided. Products with no original category remain labelled as `Unknown`.

In [13]:
products["category_name_clean"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])
)

print("Missing values in category_name_clean:")
print(products["category_name_clean"].isnull().sum())

print("\nCategories retained without English translation:")
print(
    products.loc[
        products["product_category_name_english"].isnull(),
        "category_name_clean"
    ].value_counts()
)

Missing values in category_name_clean:
0

Categories retained without English translation:
category_name_clean
Unknown                                          610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64


In [14]:
## Recheck missing values in Products
product_columns_to_check = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print("Remaining missing product attributes:")
print(products[product_columns_to_check].isnull().sum())

Remaining missing product attributes:
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


### Remaining Product Attributes

Missing values in product name length, description length, photo quantity and physical measurements were retained rather than imputed. These variables are not required for the main sales, customer segmentation and forecasting objectives of the project. Imputing values without a clear analytical requirement could introduce artificial information into the dataset. If these variables are required for a later analysis, their missing values will be handled specifically for that purpose.

## Geolocation Data Preparation

The Geolocation dataset contains a large number of completely duplicated records and multiple geographical observations for individual postcode prefixes. Exact duplicate rows are removed first. The original postcode observations are otherwise retained until an appropriate method for creating a representative location for each postcode prefix is determined.

In [15]:
geolocation_clean = geolocation.drop_duplicates().copy()

print("Geolocation rows before removing duplicates:")
print(len(geolocation))

print("\nGeolocation rows after removing exact duplicates:")
print(len(geolocation_clean))

print("\nRows removed:")
print(len(geolocation) - len(geolocation_clean))

print("\nRemaining exact duplicates:")
print(geolocation_clean.duplicated().sum())

Geolocation rows before removing duplicates:
1000163

Geolocation rows after removing exact duplicates:
738332

Rows removed:
261831

Remaining exact duplicates:
0


In [16]:
## Create one representative location per postcode prefix
geolocation_postcode = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "median"),
        geolocation_lng=("geolocation_lng", "median")
    )
)

print("Rows in postcode-level geolocation table:")
print(len(geolocation_postcode))

print("\nUnique postcode prefixes:")
print(geolocation_postcode["geolocation_zip_code_prefix"].nunique())

print("\nDuplicate postcode prefixes:")
print(geolocation_postcode["geolocation_zip_code_prefix"].duplicated().sum())

print("\nMissing coordinates:")
print(
    geolocation_postcode[
        ["geolocation_lat", "geolocation_lng"]
    ].isnull().sum()
)

Rows in postcode-level geolocation table:
19015

Unique postcode prefixes:
19015

Duplicate postcode prefixes:
0

Missing coordinates:
geolocation_lat    0
geolocation_lng    0
dtype: int64


In [17]:
## Check city/state consistency before adding them
postcode_city_counts = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix")["geolocation_city"]
    .nunique()
)

postcode_state_counts = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)

print("Postcodes associated with more than one city:")
print((postcode_city_counts > 1).sum())

print("\nMaximum number of cities associated with one postcode:")
print(postcode_city_counts.max())

print("\nPostcodes associated with more than one state:")
print((postcode_state_counts > 1).sum())

print("\nMaximum number of states associated with one postcode:")
print(postcode_state_counts.max())

Postcodes associated with more than one city:
8556

Maximum number of cities associated with one postcode:
5

Postcodes associated with more than one state:
8

Maximum number of states associated with one postcode:
2


### Geolocation Aggregation

After removing exact duplicates, multiple geolocation observations remained for individual postcode prefixes. A postcode-level table was therefore created using the median latitude and longitude for each postcode prefix. The median was selected to reduce the influence of unusually distant coordinate observations.

City and state values were not aggregated from the Geolocation dataset because many postcode prefixes were associated with multiple city labels. Customer city and state information is already available in the Customers dataset and will be retained for geographical analysis. This avoids introducing an arbitrary city-selection rule and prevents one-to-many geolocation joins from duplicating customer or transaction records.

In [18]:
## Check customer postcode coverage
customer_postcodes = customers["customer_zip_code_prefix"]

matched_postcodes = customer_postcodes.isin(
    geolocation_postcode["geolocation_zip_code_prefix"]
)

print("Total customer records:")
print(len(customers))

print("\nCustomers with matching geolocation postcode:")
print(matched_postcodes.sum())

print("\nCustomers without matching geolocation postcode:")
print((~matched_postcodes).sum())

print("\nPercentage matched:")
print(f"{matched_postcodes.mean() * 100:.2f}%")

Total customer records:
99441

Customers with matching geolocation postcode:
99163

Customers without matching geolocation postcode:
278

Percentage matched:
99.72%


## Customer Location Preparation

Customer postcode prefixes were compared with the prepared postcode-level Geolocation dataset. Most customer records could be matched, while a small number had no corresponding geolocation postcode. These customers are retained because their transaction and city/state information remains valid. A left join is used to add representative latitude and longitude coordinates without removing unmatched customers.

In [19]:
customers_clean = customers.merge(
    geolocation_postcode,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

print("Customer rows before merge:")
print(len(customers))

print("\nCustomer rows after merge:")
print(len(customers_clean))

print("\nUnique customer IDs after merge:")
print(customers_clean["customer_id"].nunique())

print("\nCustomers without coordinates:")
print(customers_clean["geolocation_lat"].isnull().sum())

Customer rows before merge:
99441

Customer rows after merge:
99441

Unique customer IDs after merge:
99441

Customers without coordinates:
278


In [20]:
## Check the Customers table for remaining problems
print("Customers Clean Validation")
print("-" * 50)

print("Duplicate customer IDs:")
print(customers_clean["customer_id"].duplicated().sum())

print("\nMissing customer_unique_id:")
print(customers_clean["customer_unique_id"].isnull().sum())

print("\nMissing customer city:")
print(customers_clean["customer_city"].isnull().sum())

print("\nMissing customer state:")
print(customers_clean["customer_state"].isnull().sum())

print("\nMissing latitude:")
print(customers_clean["geolocation_lat"].isnull().sum())

print("\nMissing longitude:")
print(customers_clean["geolocation_lng"].isnull().sum())

Customers Clean Validation
--------------------------------------------------
Duplicate customer IDs:
0

Missing customer_unique_id:
0

Missing customer city:
0

Missing customer state:
0

Missing latitude:
278

Missing longitude:
278


### Customer Data Validation

The prepared customer dataset retained all 99,441 customer records and all customer IDs remained unique after the geolocation merge. No missing values were identified in the customer identifier, city or state fields. A total of 278 customers could not be matched to latitude and longitude coordinates. These records were retained because valid customer and transaction information remains available, while the missing coordinates will only affect analyses that specifically require geographical coordinates.

## Transaction Data Preparation

A transaction-level dataset is created by combining delivered orders with their corresponding order items. The order-item level is retained because an individual order can contain multiple products. This dataset will provide the basis for sales analysis and later aggregation for forecasting.

In [21]:
transactions = delivered_order_items.merge(
    delivered_orders,
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Delivered order-item rows before merge:")
print(len(delivered_order_items))

print("\nTransaction rows after merge:")
print(len(transactions))

print("\nUnique orders in transaction dataset:")
print(transactions["order_id"].nunique())

print("\nRows without an order purchase timestamp:")
print(transactions["order_purchase_timestamp"].isnull().sum())

Delivered order-item rows before merge:
110197

Transaction rows after merge:
110197

Unique orders in transaction dataset:
96478

Rows without an order purchase timestamp:
0


In [22]:
## Add product information to the transactions
transactions = transactions.merge(
    products[
        [
            "product_id",
            "category_name_clean"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("Transaction rows after product merge:")
print(len(transactions))

print("\nUnique orders:")
print(transactions["order_id"].nunique())

print("\nMissing product IDs:")
print(transactions["product_id"].isnull().sum())

print("\nMissing clean category names:")
print(transactions["category_name_clean"].isnull().sum())

print("\nUnknown category transaction rows:")
print((transactions["category_name_clean"] == "Unknown").sum())

Transaction rows after product merge:
110197

Unique orders:
96478

Missing product IDs:
0

Missing clean category names:
0

Unknown category transaction rows:
1537


In [23]:
## Add customer information to the transactions
transactions = transactions.merge(
    customers_clean[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state",
            "geolocation_lat",
            "geolocation_lng"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Transaction rows after customer merge:")
print(len(transactions))

print("\nUnique orders:")
print(transactions["order_id"].nunique())

print("\nMissing customer IDs:")
print(transactions["customer_id"].isnull().sum())

print("\nMissing unique customer IDs:")
print(transactions["customer_unique_id"].isnull().sum())

print("\nMissing customer city:")
print(transactions["customer_city"].isnull().sum())

print("\nTransaction rows without coordinates:")
print(transactions["geolocation_lat"].isnull().sum())

Transaction rows after customer merge:
110197

Unique orders:
96478

Missing customer IDs:
0

Missing unique customer IDs:
0

Missing customer city:
0

Transaction rows without coordinates:
288


## Sales Value Preparation

The order-item dataset contains separate product price and freight values. For the main sales analysis, `sales_value` is defined using the product `price` field. Freight is retained separately so that product sales and shipping charges can be analysed independently and are not combined without justification.

In [24]:
transactions["sales_value"] = transactions["price"]

print("Sales Value Validation")
print("-" * 50)

print("Missing sales values:")
print(transactions["sales_value"].isnull().sum())

print("\nMinimum sales value:")
print(transactions["sales_value"].min())

print("\nMaximum sales value:")
print(transactions["sales_value"].max())

print("\nZero sales values:")
print((transactions["sales_value"] == 0).sum())

print("\nNegative sales values:")
print((transactions["sales_value"] < 0).sum())

print("\nTotal product sales value:")
print(round(transactions["sales_value"].sum(), 2))

print("\nTotal freight value:")
print(round(transactions["freight_value"].sum(), 2))

Sales Value Validation
--------------------------------------------------
Missing sales values:
0

Minimum sales value:
0.85

Maximum sales value:
6735.0

Zero sales values:
0

Negative sales values:
0

Total product sales value:
13221498.11

Total freight value:
2198275.64


### Payment Value Validation

Payment information is stored separately from order items and an order can contain more than one payment record. Payment values are therefore aggregated to order level before comparison with the transaction data. This prevents payment values from being duplicated when orders contain multiple items or payment methods.

In [25]:
payments_by_order = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_records=("payment_sequential", "count")
    )
)

print("Original payment rows:")
print(len(payments))

print("\nOrders represented in payments:")
print(payments_by_order["order_id"].nunique())

print("\nOrders with multiple payment records:")
print((payments_by_order["payment_records"] > 1).sum())

print("\nMaximum payment records for one order:")
print(payments_by_order["payment_records"].max())

Original payment rows:
103886

Orders represented in payments:
99440

Orders with multiple payment records:
2961

Maximum payment records for one order:
29


In [26]:
## Compare order values with payment totals
order_value_check = (
    transactions
    .groupby("order_id", as_index=False)
    .agg(
        product_value=("sales_value", "sum"),
        freight_value=("freight_value", "sum")
    )
)

order_value_check["order_total_value"] = (
    order_value_check["product_value"]
    + order_value_check["freight_value"]
)

order_value_check = order_value_check.merge(
    payments_by_order[
        ["order_id", "total_payment_value"]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

order_value_check["payment_difference"] = (
    order_value_check["total_payment_value"]
    - order_value_check["order_total_value"]
)

print("Delivered orders checked:")
print(len(order_value_check))

print("\nOrders without payment information:")
print(order_value_check["total_payment_value"].isnull().sum())

print("\nAverage absolute payment difference:")
print(
    round(
        order_value_check["payment_difference"].abs().mean(),
        4
    )
)

print("\nMaximum absolute payment difference:")
print(
    round(
        order_value_check["payment_difference"].abs().max(),
        4
    )
)

print("\nOrders with absolute difference greater than 0.05:")
print(
    (order_value_check["payment_difference"].abs() > 0.05).sum()
)

Delivered orders checked:
96478

Orders without payment information:
1

Average absolute payment difference:
0.0335

Maximum absolute payment difference:
182.81

Orders with absolute difference greater than 0.05:
256


In [27]:
## Inspect the largest payment differences
largest_payment_differences = (
    order_value_check
    .loc[
        order_value_check["payment_difference"].abs() > 0.05,
        [
            "order_id",
            "product_value",
            "freight_value",
            "order_total_value",
            "total_payment_value",
            "payment_difference"
        ]
    ]
    .copy()
)

largest_payment_differences["absolute_difference"] = (
    largest_payment_differences["payment_difference"].abs()
)

largest_payment_differences = (
    largest_payment_differences
    .sort_values("absolute_difference", ascending=False)
)

print("Orders with differences greater than 0.05:")
print(len(largest_payment_differences))

print("\nLargest 10 differences:")
display(largest_payment_differences.head(10))

Orders with differences greater than 0.05:
256

Largest 10 differences:


,order_id,product_value,freight_value,order_total_value,total_payment_value,payment_difference,absolute_difference
77799,ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1403.66,1586.47,182.81,182.81
41570,6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,287.91,406.92,119.01,119.01
42467,70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,466.93,578.82,111.89,111.89
57461,996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,587.90,664.43,76.53,76.53
42470,70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,213.15,274.84,61.69,61.69
70919,bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,242.01,303.02,61.01,61.01
65989,af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,413.17,466.97,53.80,53.80
14320,262118ce178bb3e4590a3adcf6d62e6b,119.80,57.94,177.74,126.12,-51.62,51.62
72286,bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,348.93,394.36,45.43,45.43
53097,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,254.45,293.89,39.44,39.44


In [28]:
## Inspect payment details for the largest differences
top_difference_order_ids = (
    largest_payment_differences
    .head(10)["order_id"]
    .tolist()
)

payment_details_check = payments[
    payments["order_id"].isin(top_difference_order_ids)
].sort_values(
    ["order_id", "payment_sequential"]
)

display(
    payment_details_check[
        [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value"
        ]
    ]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
85540,262118ce178bb3e4590a3adcf6d62e6b,1,credit_card,5,126.12
38060,6e5fe7366a2e1bfbf3257dba0af1267f,1,credit_card,10,406.92
25761,70b742795bc441e94a44a084b6d9ce7a,1,credit_card,20,578.82
2970,70b7e94ea46d3e8b5bc12a50186edaf0,1,credit_card,24,274.84
95180,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,1,credit_card,12,293.89
61856,996c7e73600ad3723e8627ab7bef81e4,1,credit_card,10,664.43
4954,af9ffff2ce6b3defd34fd4c78857a379,1,credit_card,10,466.97
27307,bc2c82b0ef78d2252b6176d1972db7c9,1,credit_card,21,303.02
41563,bfdb5bbb06458d600a33d61f5f287472,1,credit_card,10,394.36
359,ce6d150fb29ada17d2082f4847107665,1,credit_card,10,1586.47


### Monetary Value Treatment

Product price, freight value and payment value were treated as separate measures. Product `price` is used as the main sales value because it represents the value of products recorded at order-item level, while freight is retained separately.

Payment records were aggregated to order level before comparison because some orders contain multiple payment records. The comparison identified differences between payment totals and the combined product and freight values for a small number of delivered orders. Inspection of the largest differences did not provide sufficient evidence to treat the product prices as incorrect. Therefore, these orders were retained and payment values were not used to replace the product sales values.

This approach avoids modifying valid transaction data without clear evidence and prevents payment records from causing duplicated financial values during later joins.

In [29]:
## Validate transaction quantities
print("Order Item Validation")
print("-" * 50)

print("Minimum order_item_id:")
print(transactions["order_item_id"].min())

print("\nMaximum order_item_id:")
print(transactions["order_item_id"].max())

print("\nOrders containing more than one item:")
print(
    (transactions.groupby("order_id").size() > 1).sum()
)

print("\nMaximum number of item rows in one order:")
print(
    transactions.groupby("order_id").size().max()
)

print("\nAverage number of item rows per delivered order:")
print(
    round(transactions.groupby("order_id").size().mean(), 2)
)

Order Item Validation
--------------------------------------------------
Minimum order_item_id:
1

Maximum order_item_id:
21

Orders containing more than one item:
9635

Maximum number of item rows in one order:
21

Average number of item rows per delivered order:
1.14


### Order-Item Structure

The completed transaction dataset contains 110,197 order-item records representing 96,478 delivered orders. A total of 9,635 delivered orders contain more than one item row, with a maximum of 21 item rows in a single order. The average number of item rows per delivered order is 1.14.

The order-item level will be retained for product and sales analysis, while order-level aggregation will be used where individual orders must be counted. This distinction prevents multiple-item orders from being incorrectly counted as separate customer orders.

In [30]:
## Validate transaction uniqueness
print("Transaction Uniqueness Validation")
print("-" * 50)

print("Total transaction rows:")
print(len(transactions))

print("\nDuplicate order_id + order_item_id combinations:")
print(
    transactions.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print("\nComplete duplicate transaction rows:")
print(transactions.duplicated().sum())

Transaction Uniqueness Validation
--------------------------------------------------
Total transaction rows:
110197

Duplicate order_id + order_item_id combinations:
0

Complete duplicate transaction rows:
0


In [31]:
## Check seller relationships
print("Seller Validation")
print("-" * 50)

print("Missing seller IDs in transactions:")
print(transactions["seller_id"].isnull().sum())

print("\nUnique sellers in delivered transactions:")
print(transactions["seller_id"].nunique())

print("\nSeller IDs not found in Sellers dataset:")
print(
    (~transactions["seller_id"].isin(sellers["seller_id"])).sum()
)

print("\nDuplicate seller IDs in Sellers dataset:")
print(sellers["seller_id"].duplicated().sum())

Seller Validation
--------------------------------------------------
Missing seller IDs in transactions:
0

Unique sellers in delivered transactions:
2970

Seller IDs not found in Sellers dataset:
0

Duplicate seller IDs in Sellers dataset:
0


In [32]:
## Add seller location information
transactions = transactions.merge(
    sellers[
        [
            "seller_id",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("Transaction rows after seller merge:")
print(len(transactions))

print("\nUnique orders:")
print(transactions["order_id"].nunique())

print("\nMissing seller city:")
print(transactions["seller_city"].isnull().sum())

print("\nMissing seller state:")
print(transactions["seller_state"].isnull().sum())

print("\nDuplicate order_id + order_item_id combinations:")
print(
    transactions.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Transaction rows after seller merge:
110197

Unique orders:
96478

Missing seller city:
0

Missing seller state:
0

Duplicate order_id + order_item_id combinations:
0


Create useful date fields

## Transaction Date Preparation

Additional date fields are created from the order purchase timestamp to support time-based sales analysis and dashboard reporting. The original timestamp is retained, while separate date, year, month and day-of-week fields are added for easier aggregation and filtering.

In [33]:
transactions["purchase_date"] = (
    transactions["order_purchase_timestamp"].dt.date
)

transactions["purchase_year"] = (
    transactions["order_purchase_timestamp"].dt.year
)

transactions["purchase_month"] = (
    transactions["order_purchase_timestamp"].dt.to_period("M").astype(str)
)

transactions["purchase_day_of_week"] = (
    transactions["order_purchase_timestamp"].dt.day_name()
)

print("Missing purchase dates:")
print(transactions["purchase_date"].isnull().sum())

print("\nPurchase years:")
print(transactions["purchase_year"].value_counts().sort_index())

print("\nEarliest transaction date:")
print(transactions["purchase_date"].min())

print("\nLatest transaction date:")
print(transactions["purchase_date"].max())

Missing purchase dates:
0

Purchase years:
purchase_year
2016      317
2017    49556
2018    60324
Name: count, dtype: int64

Earliest transaction date:
2016-09-15

Latest transaction date:
2018-08-29


In [34]:
## Check the new date fields
print("Transaction Date Field Validation")
print("-" * 50)

print("Purchase months:")
print(
    transactions["purchase_month"]
    .value_counts()
    .sort_index()
)

print("\nDay-of-week values:")
print(
    transactions["purchase_day_of_week"]
    .value_counts()
)

Transaction Date Field Validation
--------------------------------------------------
Purchase months:
purchase_month
2016-09       3
2016-10     313
2016-12       1
2017-01     913
2017-02    1858
2017-03    2897
2017-04    2569
2017-05    4004
2017-06    3489
2017-07    4416
2017-08    4797
2017-09    4737
2017-10    5214
2017-11    8475
2017-12    6187
2018-01    8037
2018-02    7518
2018-03    8017
2018-04    7827
2018-05    7810
2018-06    7010
2018-07    6963
2018-08    7142
Name: count, dtype: int64

Day-of-week values:
purchase_day_of_week
Monday       17973
Tuesday      17858
Wednesday    17219
Thursday     16434
Friday       15698
Sunday       13130
Saturday     11885
Name: count, dtype: int64


In [35]:
## Check the final transaction table for missing values
transaction_missing = (
    transactions.isnull()
    .sum()
    .sort_values(ascending=False)
)

print("Missing Values in Prepared Transaction Dataset")
print("-" * 50)

print(
    transaction_missing[
        transaction_missing > 0
    ]
)

Missing Values in Prepared Transaction Dataset
--------------------------------------------------
geolocation_lat                  288
geolocation_lng                  288
order_approved_at                 15
order_delivered_customer_date      8
order_delivered_carrier_date       2
dtype: int64


### Remaining Missing Values

A final missing-value check identified a small number of missing values in the prepared transaction dataset. These consist of customer geographical coordinates and several order processing or delivery timestamps.

The affected records were retained because the missing fields do not prevent the transactions from being used for the main sales, customer and forecasting analyses. Missing geographical coordinates will only affect analyses requiring map coordinates. Similarly, records with missing delivery timestamps will be handled specifically if delivery-performance measures are calculated.

This approach avoids removing otherwise valid completed transactions because of missing values in fields that are not required for the main project objectives.

In [36]:
## Final transaction validation
print("Final Transaction Dataset Validation")
print("-" * 50)

print("Total transaction rows:")
print(len(transactions))

print("\nUnique delivered orders:")
print(transactions["order_id"].nunique())

print("\nUnique customers:")
print(transactions["customer_unique_id"].nunique())

print("\nUnique products:")
print(transactions["product_id"].nunique())

print("\nUnique sellers:")
print(transactions["seller_id"].nunique())

print("\nDuplicate order-item combinations:")
print(
    transactions.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print("\nMissing sales values:")
print(transactions["sales_value"].isnull().sum())

print("\nNegative sales values:")
print((transactions["sales_value"] < 0).sum())

print("\nTotal product sales value:")
print(round(transactions["sales_value"].sum(), 2))

print("\nTransaction date range:")
print(
    transactions["purchase_date"].min(),
    "to",
    transactions["purchase_date"].max()
)

print("\nDataset shape:")
print(transactions.shape)

Final Transaction Dataset Validation
--------------------------------------------------
Total transaction rows:
110197

Unique delivered orders:
96478

Unique customers:
93358

Unique products:
32216

Unique sellers:
2970

Duplicate order-item combinations:
0

Missing sales values:
0

Negative sales values:
0

Total product sales value:
13221498.11

Transaction date range:
2016-09-15 to 2018-08-29

Dataset shape:
(110197, 27)


## Final Transaction Dataset Validation

The prepared transaction dataset contains 110,197 order-item records representing 96,478 delivered orders and 93,358 unique customers. It includes 32,216 products and 2,970 sellers.

No duplicate order and item combinations were identified, and the sales value field contains no missing, zero or negative values. The total product sales value is 13,221,498.11. The completed transaction period extends from 15 September 2016 to 29 August 2018.

The prepared transaction dataset therefore preserves the valid completed sales records while maintaining the distinction between individual order items and unique orders. Remaining missing geographical coordinates and delivery timestamps have been retained because they do not prevent the main sales, customer and forecasting analyses.

In [37]:
## Create the cleaned output files
transactions.to_csv(
    cleaned_data_path / "transactions_clean.csv",
    index=False
)

print("transactions_clean.csv saved successfully.")

print("\nRows exported:")
print(len(transactions))

print("\nColumns exported:")
print(len(transactions.columns))

print("\nSaved to:")
print(cleaned_data_path / "transactions_clean.csv")

transactions_clean.csv saved successfully.

Rows exported:
110197

Columns exported:
27

Saved to:
..\02_Cleaned_Data\transactions_clean.csv


In [38]:
## Export the cleaned customer table
customers_clean = customers_clean.drop(
    columns=["geolocation_zip_code_prefix"]
).rename(
    columns={
        "geolocation_lat": "customer_lat",
        "geolocation_lng": "customer_lng"
    }
)

customers_clean.to_csv(
    cleaned_data_path / "customers_clean.csv",
    index=False
)

print("customers_clean.csv saved successfully.")

print("\nRows exported:")
print(len(customers_clean))

print("\nColumns exported:")
print(len(customers_clean.columns))

print("\nUnique customer IDs:")
print(customers_clean["customer_id"].nunique())

print("\nMissing customer coordinates:")
print(customers_clean["customer_lat"].isnull().sum())

print("\nSaved to:")
print(cleaned_data_path / "customers_clean.csv")


customers_clean.csv saved successfully.

Rows exported:
99441

Columns exported:
7

Unique customer IDs:
99441

Missing customer coordinates:
278

Saved to:
..\02_Cleaned_Data\customers_clean.csv


In [39]:
## Export the prepared product table
products.to_csv(
    cleaned_data_path / "products_clean.csv",
    index=False
)

print("products_clean.csv saved successfully.")

print("\nRows exported:")
print(len(products))

print("\nColumns exported:")
print(len(products.columns))

print("\nUnique product IDs:")
print(products["product_id"].nunique())

print("\nMissing clean category names:")
print(products["category_name_clean"].isnull().sum())

print("\nProducts labelled Unknown:")
print((products["category_name_clean"] == "Unknown").sum())

print("\nSaved to:")
print(cleaned_data_path / "products_clean.csv")

products_clean.csv saved successfully.

Rows exported:
32951

Columns exported:
11

Unique product IDs:
32951

Missing clean category names:
0

Products labelled Unknown:
610

Saved to:
..\02_Cleaned_Data\products_clean.csv


In [40]:
## Export the prepared orders table
orders.to_csv(
    cleaned_data_path / "orders_clean.csv",
    index=False
)

print("orders_clean.csv saved successfully.")

print("\nRows exported:")
print(len(orders))

print("\nColumns exported:")
print(len(orders.columns))

print("\nUnique order IDs:")
print(orders["order_id"].nunique())

print("\nOrder statuses:")
print(orders["order_status"].value_counts())

print("\nSaved to:")
print(cleaned_data_path / "orders_clean.csv")

orders_clean.csv saved successfully.

Rows exported:
99441

Columns exported:
8

Unique order IDs:
99441

Order statuses:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Saved to:
..\02_Cleaned_Data\orders_clean.csv


In [41]:
## Export the prepared postcode geolocation table
geolocation_postcode.to_csv(
    cleaned_data_path / "geolocation_postcode_clean.csv",
    index=False
)

print("geolocation_postcode_clean.csv saved successfully.")

print("\nRows exported:")
print(len(geolocation_postcode))

print("\nUnique postcode prefixes:")
print(
    geolocation_postcode[
        "geolocation_zip_code_prefix"
    ].nunique()
)

print("\nDuplicate postcode prefixes:")
print(
    geolocation_postcode[
        "geolocation_zip_code_prefix"
    ].duplicated().sum()
)

print("\nMissing latitude:")
print(geolocation_postcode["geolocation_lat"].isnull().sum())

print("\nMissing longitude:")
print(geolocation_postcode["geolocation_lng"].isnull().sum())

print("\nSaved to:")
print(
    cleaned_data_path / "geolocation_postcode_clean.csv"
)

geolocation_postcode_clean.csv saved successfully.

Rows exported:
19015

Unique postcode prefixes:
19015

Duplicate postcode prefixes:
0

Missing latitude:
0

Missing longitude:
0

Saved to:
..\02_Cleaned_Data\geolocation_postcode_clean.csv


In [42]:
## Export the aggregated payment table
payments_by_order.to_csv(
    cleaned_data_path / "payments_by_order_clean.csv",
    index=False
)

print("payments_by_order_clean.csv saved successfully.")

print("\nRows exported:")
print(len(payments_by_order))

print("\nUnique order IDs:")
print(payments_by_order["order_id"].nunique())

print("\nDuplicate order IDs:")
print(payments_by_order["order_id"].duplicated().sum())

print("\nMissing total payment values:")
print(payments_by_order["total_payment_value"].isnull().sum())

print("\nSaved to:")
print(cleaned_data_path / "payments_by_order_clean.csv")

payments_by_order_clean.csv saved successfully.

Rows exported:
99440

Unique order IDs:
99440

Duplicate order IDs:
0

Missing total payment values:
0

Saved to:
..\02_Cleaned_Data\payments_by_order_clean.csv


In [43]:
## Check the cleaned-data folder
cleaned_files = sorted(cleaned_data_path.glob("*.csv"))

print("Cleaned Data Files")
print("-" * 50)

for file in cleaned_files:
    file_size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name}: {file_size_mb:.2f} MB")

print("\nTotal cleaned CSV files:")
print(len(cleaned_files))

Cleaned Data Files
--------------------------------------------------
customers_clean.csv: 11.91 MB
geolocation_postcode_clean.csv: 0.82 MB
orders_clean.csv: 15.84 MB
payments_by_order_clean.csv: 4.03 MB
products_clean.csv: 3.57 MB
transactions_clean.csv: 44.03 MB

Total cleaned CSV files:
6


## Data Cleaning and Preparation Summary

The Olist datasets were cleaned and prepared while preserving the original raw files. Date and timestamp fields were converted to appropriate datetime formats, and delivered orders were separated for completed sales analysis.

Missing values were assessed according to their analytical importance rather than applying blanket deletion or imputation. Missing product categories were labelled as `Unknown`, while valid original category names were retained where English translations were unavailable. Missing product attributes that are not required for the main project objectives were left unchanged.

The Geolocation dataset contained 261,831 exact duplicate rows and multiple observations for individual postcode prefixes. Exact duplicates were removed and representative median latitude and longitude coordinates were calculated for each postcode prefix. This produced 19,015 unique postcode-level records and prevented one-to-many geographical joins from duplicating transaction data.

The prepared transaction dataset contains 110,197 order-item records representing 96,478 delivered orders and 93,358 unique customers. Product price was retained as the main sales measure, while freight and payment values were treated separately. No duplicate order-item combinations, missing sales values or negative sales values were identified. The total product sales value is 13,221,498.11.

Six prepared datasets were exported to the `02_Cleaned_Data` folder for later SQL, Python and Power BI analysis:

- `transactions_clean.csv`
- `customers_clean.csv`
- `products_clean.csv`
- `orders_clean.csv`
- `geolocation_postcode_clean.csv`
- `payments_by_order_clean.csv`

The original raw datasets remain unchanged in the `01_Raw_Data` folder, providing a clear separation between source data and prepared analytical data.